# Hierarchical Transformer with Whitespace Boundaries

This notebook implements a small-scale hierarchical autoregressive transformer inspired by Neitemeier et al. (ICLR 2025). It uses UTF-8 bytes as the base alphabet, splits text on whitespace boundaries with whitespace attached to the previous word, encodes each word with a small byte-level encoder, processes the resulting word embeddings with a causal backbone, and decodes the next word byte-by-byte.

In [ ]:
!pip install -q datasets tqdm matplotlib

In [ ]:
import math
import re
import time
from dataclasses import dataclass
from typing import Optional

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from metrics import evaluate_all_metrics, whitespace_boundary_positions
from results_utils import save_experiment_artifacts
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm


if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


@dataclass
class Config:
    train_stories: int = 5000
    val_stories: int = 2000
    max_word_bytes: int = 24
    word_block_size: int = 64
    batch_size: int = 16
    char_d_model: int = 128
    word_d_model: int = 256
    char_heads: int = 4
    word_heads: int = 4
    char_layers: int = 2
    word_layers: int = 4
    dropout: float = 0.1
    learning_rate: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    epochs: int = 3
    eval_every: int = 300
    val_max_batches: int = 20
    max_train_steps: Optional[int] = 15000
    sample_prompt: str = "Once upon a time "
    sample_words: int = 40


cfg = Config()
cfg

## Paper Alignment Notes

The ICLR 2025 paper uses UTF-8 bytes as the base alphabet and splits text at Unicode whitespace characters, appending the whitespace to the previous word. This notebook follows that splitting rule directly. At this project scale, the implementation is intentionally compact, but the data flow matches the paper's encoder/backbone/decoder factorization.

In [ ]:
ds = load_dataset("roneneldan/TinyStories")

train_split = ds["train"].select(range(min(cfg.train_stories, len(ds["train"]))))
val_split = ds["validation"].select(range(min(cfg.val_stories, len(ds["validation"]))))

train_texts = train_split["text"]
val_texts = val_split["text"]

print(f"Train stories: {len(train_texts):,}")
print(f"Validation stories: {len(val_texts):,}")

## Whitespace-Based Word Splitting

In [ ]:
PAD_BYTE = 256
WORD_START = 257
WORD_END = 258
BYTE_VOCAB_SIZE = 259


def split_with_attached_whitespace(text):
    parts = re.findall(r"\S+\s*|\s+", text)
    words = []
    for part in parts:
        if part.isspace():
            if words:
                words[-1] += part
            else:
                words.append(part)
        else:
            words.append(part)
    return [word for word in words if word]


def word_to_bytes(word):
    return list(word.encode("utf-8"))


def bytes_to_word(byte_ids):
    raw = bytes([b for b in byte_ids if 0 <= int(b) <= 255])
    return raw.decode("utf-8", errors="ignore")


def encode_word(word, max_word_bytes):
    byte_ids = word_to_bytes(word)[: max_word_bytes - 1]
    decoder_input = [WORD_START] + byte_ids
    decoder_target = byte_ids + [WORD_END]

    decoder_input = decoder_input[:max_word_bytes]
    decoder_target = decoder_target[:max_word_bytes]

    input_len = len(decoder_input)
    target_len = len(decoder_target)

    decoder_input = decoder_input + [PAD_BYTE] * (max_word_bytes - input_len)
    decoder_target = decoder_target + [PAD_BYTE] * (max_word_bytes - target_len)
    target_mask = [1] * target_len + [0] * (max_word_bytes - target_len)

    return {
        "encoder_bytes": [WORD_START] + byte_ids,
        "decoder_input": decoder_input,
        "decoder_target": decoder_target,
        "target_mask": target_mask,
    }


sample_words = split_with_attached_whitespace(cfg.sample_prompt)
sample_words

In [ ]:
def build_word_sequences(texts, max_word_bytes):
    documents = []
    total_words = 0

    for text in texts:
        words = split_with_attached_whitespace(text)
        if len(words) < 2:
            continue

        encoded_words = [encode_word(word, max_word_bytes) for word in words]
        documents.append(encoded_words)
        total_words += len(encoded_words)

    return documents, total_words


train_docs, train_total_words = build_word_sequences(train_texts, cfg.max_word_bytes)
val_docs, val_total_words = build_word_sequences(val_texts, cfg.max_word_bytes)

print(f"Train docs: {len(train_docs):,} | train words: {train_total_words:,}")
print(f"Val docs: {len(val_docs):,} | val words: {val_total_words:,}")
print(train_docs[0][0])

## Dataset

In [ ]:
class HierarchicalWordDataset(Dataset):
    def __init__(self, documents, word_block_size, max_word_bytes):
        self.documents = documents
        self.word_block_size = word_block_size
        self.max_word_bytes = max_word_bytes
        self.samples = []

        for doc_idx, doc in enumerate(documents):
            if len(doc) <= word_block_size:
                continue
            for start in range(0, len(doc) - word_block_size):
                self.samples.append((doc_idx, start))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        doc_idx, start = self.samples[idx]
        doc = self.documents[doc_idx]
        context_words = doc[start : start + self.word_block_size]
        next_words = doc[start + 1 : start + self.word_block_size + 1]

        encoder_inputs = [word["encoder_bytes"] for word in context_words]
        decoder_inputs = [word["decoder_input"] for word in next_words]
        decoder_targets = [word["decoder_target"] for word in next_words]
        target_masks = [word["target_mask"] for word in next_words]

        return {
            "encoder_inputs": encoder_inputs,
            "decoder_inputs": torch.tensor(decoder_inputs, dtype=torch.long),
            "decoder_targets": torch.tensor(decoder_targets, dtype=torch.long),
            "target_masks": torch.tensor(target_masks, dtype=torch.float32),
        }


def collate_batch(batch):
    batch_size = len(batch)
    word_block_size = len(batch[0]["encoder_inputs"])
    max_encoder_len = max(len(word) for item in batch for word in item["encoder_inputs"])

    encoder_tokens = torch.full((batch_size, word_block_size, max_encoder_len), PAD_BYTE, dtype=torch.long)
    encoder_mask = torch.zeros((batch_size, word_block_size, max_encoder_len), dtype=torch.bool)

    for batch_idx, item in enumerate(batch):
        for word_idx, word in enumerate(item["encoder_inputs"]):
            word_tensor = torch.tensor(word, dtype=torch.long)
            encoder_tokens[batch_idx, word_idx, : len(word)] = word_tensor
            encoder_mask[batch_idx, word_idx, : len(word)] = True

    return {
        "encoder_tokens": encoder_tokens,
        "encoder_mask": encoder_mask,
        "decoder_inputs": torch.stack([item["decoder_inputs"] for item in batch]),
        "decoder_targets": torch.stack([item["decoder_targets"] for item in batch]),
        "target_masks": torch.stack([item["target_masks"] for item in batch]),
    }


train_dataset = HierarchicalWordDataset(train_docs, cfg.word_block_size, cfg.max_word_bytes)
val_dataset = HierarchicalWordDataset(val_docs, cfg.word_block_size, cfg.max_word_bytes)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=pin_memory,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=pin_memory,
    collate_fn=collate_batch,
)

print(f"Train windows: {len(train_dataset):,}")
print(f"Validation windows: {len(val_dataset):,}")

## Hierarchical Model

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=128):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal_mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask.view(1, 1, block_size, block_size), persistent=False)

    def forward(self, x):
        batch_size, seq_len, channels = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.n_heads, self.head_dim).transpose(1, 2)
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = scores.masked_fill(~self.causal_mask[:, :, :seq_len, :seq_len], float("-inf"))
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)
        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, channels)
        out = self.out_proj(out)
        return self.resid_dropout(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, block_size=128, causal=True):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        if causal:
            self.attn = CausalSelfAttention(d_model, n_heads, dropout=dropout, block_size=block_size)
        else:
            self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.causal = causal
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, key_padding_mask=None):
        h = self.ln1(x)
        if self.causal:
            x = x + self.attn(h)
        else:
            attn_out, _ = self.attn(h, h, h, key_padding_mask=key_padding_mask, need_weights=False)
            x = x + attn_out
        x = x + self.ffn(self.ln2(x))
        return x


class WordEncoder(nn.Module):
    def __init__(self, byte_vocab_size, d_model, n_heads, n_layers, max_word_bytes, dropout=0.1):
        super().__init__()
        self.byte_embedding = nn.Embedding(byte_vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_word_bytes + 1, d_model)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, dropout=dropout, block_size=max_word_bytes + 1, causal=False) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)

    def forward(self, tokens, mask):
        positions = torch.arange(tokens.size(-1), device=tokens.device)
        x = self.byte_embedding(tokens) + self.position_embedding(positions)[None, None, :, :]
        batch_size, num_words, seq_len, dim = x.shape
        x = x.view(batch_size * num_words, seq_len, dim)
        flat_mask = (~mask).view(batch_size * num_words, seq_len)

        for block in self.blocks:
            x = block(x, key_padding_mask=flat_mask)

        x = self.ln_f(x)
        word_embeddings = x[:, 0, :]
        return word_embeddings.view(batch_size, num_words, dim)


class WordBackbone(nn.Module):
    def __init__(self, input_dim, d_model, n_heads, n_layers, block_size, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.position_embedding = nn.Embedding(block_size, d_model)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, dropout=dropout, block_size=block_size, causal=True) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.output_proj = nn.Linear(d_model, input_dim)

    def forward(self, word_embeddings):
        batch_size, seq_len, _ = word_embeddings.shape
        positions = torch.arange(seq_len, device=word_embeddings.device)
        x = self.input_proj(word_embeddings) + self.position_embedding(positions)[None, :, :]

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        return self.output_proj(x)


class ByteDecoder(nn.Module):
    def __init__(self, byte_vocab_size, d_model, n_heads, n_layers, max_word_bytes, dropout=0.1):
        super().__init__()
        self.byte_embedding = nn.Embedding(byte_vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_word_bytes, d_model)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, dropout=dropout, block_size=max_word_bytes, causal=True) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, byte_vocab_size)

    def forward(self, predictive_embeddings, decoder_inputs):
        batch_size, num_words, seq_len = decoder_inputs.shape
        positions = torch.arange(seq_len, device=decoder_inputs.device)
        token_emb = self.byte_embedding(decoder_inputs)
        pos_emb = self.position_embedding(positions)[None, None, :, :]
        x = token_emb + pos_emb + predictive_embeddings[:, :, None, :]
        x = x.view(batch_size * num_words, seq_len, -1)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.lm_head(x)
        return logits.view(batch_size, num_words, seq_len, -1)


class HierarchicalWhitespaceTransformer(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.max_word_bytes = cfg.max_word_bytes
        self.word_block_size = cfg.word_block_size
        self.encoder = WordEncoder(
            BYTE_VOCAB_SIZE,
            cfg.char_d_model,
            cfg.char_heads,
            cfg.char_layers,
            cfg.max_word_bytes,
            dropout=cfg.dropout,
        )
        self.backbone = WordBackbone(
            cfg.char_d_model,
            cfg.word_d_model,
            cfg.word_heads,
            cfg.word_layers,
            cfg.word_block_size,
            dropout=cfg.dropout,
        )
        self.decoder = ByteDecoder(
            BYTE_VOCAB_SIZE,
            cfg.char_d_model,
            cfg.char_heads,
            cfg.char_layers,
            cfg.max_word_bytes,
            dropout=cfg.dropout,
        )
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, encoder_tokens, encoder_mask, decoder_inputs, decoder_targets=None, target_masks=None):
        word_embeddings = self.encoder(encoder_tokens, encoder_mask)
        predictive_embeddings = self.backbone(word_embeddings)
        logits = self.decoder(predictive_embeddings, decoder_inputs)

        loss = None
        if decoder_targets is not None and target_masks is not None:
            flat_logits = logits.view(-1, BYTE_VOCAB_SIZE)
            flat_targets = decoder_targets.view(-1)
            flat_mask = target_masks.view(-1)

            token_losses = F.cross_entropy(flat_logits, flat_targets, reduction="none")
            loss = (token_losses * flat_mask).sum() / flat_mask.sum()

        return logits, loss

    @torch.no_grad()
    def generate_next_word(self, context_words, device, max_word_bytes, temperature=1.0, top_k=20):
        encoded_context = [encode_word(word, max_word_bytes)["encoder_bytes"] for word in context_words[-self.word_block_size :]]
        seq_len = max(len(word) for word in encoded_context)
        encoder_tokens = torch.full((1, len(encoded_context), seq_len), PAD_BYTE, dtype=torch.long, device=device)
        encoder_mask = torch.zeros((1, len(encoded_context), seq_len), dtype=torch.bool, device=device)

        for idx, word in enumerate(encoded_context):
            encoder_tokens[0, idx, : len(word)] = torch.tensor(word, dtype=torch.long, device=device)
            encoder_mask[0, idx, : len(word)] = True

        word_embeddings = self.encoder(encoder_tokens, encoder_mask)
        predictive_embedding = self.backbone(word_embeddings)[:, -1:, :]

        generated = [WORD_START]
        for _ in range(max_word_bytes):
            decoder_inputs = torch.tensor(generated, dtype=torch.long, device=device).view(1, 1, -1)
            logits = self.decoder(predictive_embedding, decoder_inputs)
            next_logits = logits[0, 0, -1] / temperature
            if top_k is not None:
                values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits[next_logits < values[-1]] = float("-inf")
            probs = F.softmax(next_logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1).item()
            if next_id == WORD_END:
                break
            generated.append(next_id)

        return bytes_to_word(generated[1:])


def count_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

## Training

In [ ]:
from experiment_utils import count_parameters, pick_device, train_model


def target_token_count(batch):
    return batch["target_masks"].sum().item()


device = pick_device()
print(f"Using device: {device}")

In [ ]:
model = HierarchicalWhitespaceTransformer(cfg).to(device)
print(f"Model parameters: {count_parameters(model) / 1e6:.2f}M")

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
    checkpoint_path="hierarchical_whitespace_best.pt",
    token_count_fn=target_token_count,
)

history["best_val_loss"]

In [ ]:
if history["steps"]:
    plt.figure(figsize=(10, 4))
    plt.plot(history["steps"], history["train_loss"], label="train loss")
    plt.plot(history["steps"], history["val_loss"], label="val loss")
    plt.xlabel("step")
    plt.ylabel("byte cross-entropy")
    plt.title("Hierarchical Whitespace Transformer Training Curve")
    plt.legend()
    plt.show()
else:
    print("No intermediate evaluations were recorded. Lower cfg.eval_every if you want a curve.")

## Generation

In [ ]:
model.load_state_dict(torch.load("hierarchical_whitespace_best.pt", map_location=device))
model.eval()

generated_words = split_with_attached_whitespace(cfg.sample_prompt)
for _ in range(cfg.sample_words):
    next_word = model.generate_next_word(
        generated_words,
        device=device,
        max_word_bytes=cfg.max_word_bytes,
        temperature=1.0,
        top_k=20,
    )
    generated_words.append(next_word)

sample_text = "".join(generated_words)
print(sample_text)

## Evaluation and Artifact Export

In [ ]:
ood_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
ood_texts = [text for text in ood_dataset["text"] if text.strip()][:200]


def byte_nll_fn(texts):
    model.eval()
    total_nll = 0.0
    total_bytes = 0

    with torch.no_grad():
        for text in texts:
            words = split_with_attached_whitespace(text)
            if len(words) < 2:
                continue

            encoded_words = [encode_word(word, cfg.max_word_bytes) for word in words]

            for start in range(0, len(encoded_words) - 1, cfg.word_block_size):
                context_words = encoded_words[start : start + cfg.word_block_size]
                next_words = encoded_words[start + 1 : start + cfg.word_block_size + 1]
                paired_len = min(len(context_words), len(next_words))
                if paired_len == 0:
                    continue
                context_words = context_words[:paired_len]
                next_words = next_words[:paired_len]

                max_encoder_len = max(len(word["encoder_bytes"]) for word in context_words)
                encoder_tokens = torch.full((1, len(context_words), max_encoder_len), PAD_BYTE, dtype=torch.long, device=device)
                encoder_mask = torch.zeros((1, len(context_words), max_encoder_len), dtype=torch.bool, device=device)

                for word_idx, word in enumerate(context_words):
                    encoded = torch.tensor(word["encoder_bytes"], dtype=torch.long, device=device)
                    encoder_tokens[0, word_idx, : len(encoded)] = encoded
                    encoder_mask[0, word_idx, : len(encoded)] = True

                decoder_inputs = torch.tensor([[word["decoder_input"] for word in next_words]], dtype=torch.long, device=device)
                decoder_targets = torch.tensor([[word["decoder_target"] for word in next_words]], dtype=torch.long, device=device)

                logits, _ = model(encoder_tokens, encoder_mask, decoder_inputs)
                flat_logits = logits.view(-1, BYTE_VOCAB_SIZE)
                flat_targets = decoder_targets.view(-1)
                token_losses = F.cross_entropy(flat_logits, flat_targets, reduction="none")
                byte_mask = (flat_targets >= 0) & (flat_targets < 256)

                total_nll += token_losses[byte_mask].sum().item()
                total_bytes += byte_mask.sum().item()

    return total_nll / total_bytes


boundary_sets = [whitespace_boundary_positions(text) for text in val_texts[:200]]
metrics = evaluate_all_metrics(
    in_domain_texts=val_texts,
    ood_texts=ood_texts,
    byte_nll_fn=byte_nll_fn,
    predicted_boundary_sets=boundary_sets,
    gold_boundary_sets=boundary_sets,
    max_items=200,
    noisy_char_error_rate=0.05,
    noise_seed=42,
)
metrics

In [ ]:
save_experiment_artifacts(
    "hierarchical_whitespace",
    metrics=metrics,
    config=cfg,
    sample_text=sample_text,
    extra={"ood_preview": ood_texts[:5]},
)

print("Saved hierarchical_whitespace_metrics.json, hierarchical_whitespace_config.json, hierarchical_whitespace_sample.txt")